# Unidad 5 — Sistemas Multi-Agente Modernos
## Notebook 03: CrewAI — Sistemas Multi-Agente con Roles `[Intermedio ★★★☆☆]`

**Duración estimada:** 5-6 horas  
**Entorno:** `ia_multiagente` (Python 3.11)  
**Modelo por defecto:** Gemini 2.0 Flash · Alternativa: Ollama Llama 3.2  
**Costo estimado:** ~$0.03 con Gemini Flash · $0 con Ollama  
**Skill que se crea:** `external_skills/evaluation/output_scorer.py`

---

## Tabla de Contenidos

1. [Warm-Up y Repaso de U5_02](#1-warmup)
2. [¿Por qué CrewAI? El modelo de tripulación](#2-por-que-crewai)
3. [Primitivas de CrewAI: Agent, Task, Crew, Process](#3-primitivas)
   - 3.1 Agent — Role, Goal, Backstory
   - 3.2 Task — description, expected_output
   - 3.3 Crew y Process: Sequential vs Hierarchical
4. [Tools en CrewAI](#4-tools)
   - 4.1 Tools nativas de CrewAI
   - 4.2 Bridging: tools de LangChain en CrewAI
5. [Process Hierarchical: Manager + Delegates](#5-hierarchical)
6. [Skill: Output Scorer — Evaluación Automática](#6-output-scorer)
7. [Proyecto: Equipo de Investigación en Nanotecnología](#7-proyecto)
   - 7.1 Definir el equipo
   - 7.2 Ejecutar la misión
   - 7.3 Evaluar outputs con la skill
8. [Resumen y Criterios de Evaluación](#8-resumen)

---

## Objetivos de Aprendizaje

Al completar esta notebook serás capaz de:
- Definir `Agent`, `Task` y `Crew` con configuración completa de roles y backstories
- Comparar el `Process.sequential` vs `Process.hierarchical` con un ejemplo ejecutado
- Reutilizar tools de LangChain dentro de agentes CrewAI
- Evaluar la calidad de un output multi-agente usando la skill `output_scorer`
- Construir un equipo de investigación de 4 agentes para una tarea de nanotecnología

**Prerequisito:** U5_01 (tools, skills), U5_02 (LangGraph, routing)

---
## 1. Warm-Up y Repaso de U5_02 <a id='1-warmup'></a>

In [1]:
# ============================================================
# WARM-UP: Entorno y Dependencias — U5_03
# ============================================================
import subprocess, sys, os
from pathlib import Path

def check_install(package, import_name=None):
    name = import_name or package.split('==')[0].replace('-', '_')
    try:
        __import__(name)
        print(f"  OK: {package}")
    except ImportError:
        print(f"  Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"  Instalado: {package}")

packages = [
    ("crewai==0.80.0",                   "crewai"),
    ("crewai-tools==0.14.0",             "crewai_tools"),
    ("langchain-openai==0.3.12",         "langchain_openai"),
    ("langchain-google-genai==2.1.4",    "langchain_google_genai"),
    # ("langchain-ollama==0.3.2",          "langchain_ollama"),
    ("python-dotenv==1.1.0",             "dotenv"),
]

print("Verificando paquetes...")
for pkg, imp in packages:
    check_install(pkg, imp)

from dotenv import load_dotenv
load_dotenv(override=True)

# project root
project_root = Path.cwd()
for _ in range(5):
    if (project_root / "external_skills").exists():
        break
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Detectar backends disponibles
google_key     = os.environ.get("GOOGLE_API_KEY")
# openai_key     = os.environ.get("OPENAI_API_KEY")
openrouter_key = os.environ.get("OPENROUTER_API_KEY")

backend = (
    "OpenRouter"     if openrouter_key else
    "Gemini"         if google_key     else
    # "OpenAI"         if openai_key     else
    # "Ollama (local)"
    "Sin backend — configura OPENROUTER_API_KEY o GOOGLE_API_KEY en .env"
)

print(f"\nProject root: {project_root}")
print(f"Backend LLM activo: {backend}")
print("Warm-up completado.")


Verificando paquetes...
  OK: crewai==0.80.0


c:\Users\UCEMICH\anaconda3\envs\ia_nano\Lib\site-packages\crewai_tools\tools\rag\rag_tool.py:9: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class Adapter(BaseModel, ABC):


  OK: crewai-tools==0.14.0
  OK: langchain-openai==0.3.12
  OK: langchain-google-genai==2.1.4
  OK: python-dotenv==1.1.0

Project root: d:\Users\UCEMICH\Desktop\antigravity projects\IA NANOTECNOLOGIA\Antigravity-Nano-Research-Multiagentic-Core
Backend LLM activo: OpenRouter
Warm-up completado.


In [1]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

# Asegurar que el project_root (con external_skills) está en sys.path
# (redundante con cell 1, pero hace esta celda auto-contenida)
_root = Path.cwd()
for _ in range(6):
    if (_root / "external_skills").exists():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
    _root = _root.parent

# Configurar LLM compatible con CrewAI 0.80.0 — prioridad: OpenRouter → Gemini
# Se usa crewai.LLM (no wrappers LangChain) para que litellm enrute correctamente
from crewai import LLM

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if OPENROUTER_API_KEY:
    OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "google/gemini-2.5-flash")
    llm = LLM(
        model=f"openrouter/{OPENROUTER_MODEL}",
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
        temperature=0.2,
    )
    print(f"LLM: OpenRouter — {OPENROUTER_MODEL}")
elif GOOGLE_API_KEY:
    llm = LLM(
        model="gemini/gemini-2.0-flash",
        api_key=GOOGLE_API_KEY,
        temperature=0.2,
    )
    print("LLM: Gemini 2.0 Flash (Google AI)")
else:
    raise EnvironmentError("Configura OPENROUTER_API_KEY o GOOGLE_API_KEY en .env")

# Repaso U5_02: verificar skills disponibles
from external_skills.agent_warmup.context_loader import warm_up, SKILL_METADATA
from external_skills.routing.task_classifier import classify
print(f"\nSkills disponibles: context_loader v{SKILL_METADATA['version']}, task_classifier OK")


LLM: OpenRouter — google/gemini-2.5-flash

Skills disponibles: context_loader v1.0.0, task_classifier OK


---
## 2. ¿Por qué CrewAI? El Modelo de Tripulación <a id='2-por-que-crewai'></a>

LangGraph es excelente para flujos de trabajo complejos y control fino del estado. Pero hay casos de uso donde pensamos **naturalmente en roles humanos**: un equipo de investigación tiene un investigador senior, un analista de datos, un redactor técnico y un revisor de calidad. CrewAI mapea directamente a esa intuición.

**CrewAI vs LangGraph — cuándo usar cada uno:**

| Criterio | CrewAI | LangGraph |
|---------|--------|----------|
| **Paradigma mental** | Equipo de personas con roles | Máquina de estados / flujo de proceso |
| **Fortaleza** | Colaboración natural entre agentes con personalidades | Control fino del flujo, ciclos complejos, state management |
| **Curva de aprendizaje** | Más baja — intuitivo para no-ingenieros | Más alta — requiere pensar en grafos |
| **Delegación** | Process.hierarchical: Manager asigna dinamicamente | Manual: nodos explícitos en el grafo |
| **Observabilidad** | CrewAI+ dashboard (cloud) | LangSmith (tracing) |
| **Personalización** | Media | Alta |
| **Producción enterprise** | CrewAI Enterprise / CrewAI+ | LangGraph Platform |

La regla práctica: **si tu problema se describe naturalmente como "un equipo donde X hace esto y Y hace aquello", usa CrewAI. Si necesitas control de flujo complejo con ciclos y condiciones intrincadas, usa LangGraph.**

Y se pueden combinar: un LangGraph puede tener nodos que internamente son Crews de CrewAI.

---
## 3. Primitivas de CrewAI: Agent, Task, Crew, Process <a id='3-primitivas'></a>

### 3.1 Agent — Role, Goal, Backstory

Un `Agent` en CrewAI tiene tres campos fundamentales que determinan su comportamiento:

- **role:** Título del cargo — lo que ES el agente. Ej: "Investigador Senior en IA"
- **goal:** Objetivo principal que guía sus decisiones. Ej: "Encontrar papers relevantes y sintetizar el estado del arte"
- **backstory:** Historia de fondo que establece expertise, estilo y perspectiva. Esto va al system prompt y tiene un impacto dramático en la calidad de las respuestas

La **backstory** es donde se inyecta el equivalente al warm-up de skill de U5_01. Cuanto más específica y detallada es la backstory, mejor desempeña el agente su rol.

In [2]:
# ============================================================
# WARM-UP: Sección 3 — Imports de CrewAI
# ============================================================
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool

print("CrewAI imports OK")

CrewAI imports OK


In [9]:
# ────────────────────────────────────────────────────────────
# Output esperado: dos agentes con roles y backstories distintas
# ────────────────────────────────────────────────────────────


# ============================================================
# CONFIGURACION CON OPENROUTER - MODELOS GRATUITOS
# ============================================================
import os
import logging
from dotenv import load_dotenv
from crewai import Agent, LLM

# Suprimir warnings de LiteLLM
logging.getLogger("litellm").setLevel(logging.ERROR)

# Cargar variables de entorno
load_dotenv(override=True)

# Verificar que existe la API key
openrouter_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_key:
    raise ValueError("OPENROUTER_API_KEY no encontrada en .env")

# ============================================================
# CONFIGURACION DE MODELOS GRATUITOS CON OPENROUTER
# Se usa crewai.LLM con prefijo "openrouter/" para que litellm
# enrute correctamente a través de la API de OpenRouter
#
# Nota: google/gemini-2.0-flash-exp fue retirado de OpenRouter.
# Se usa google/gemini-2.0-flash-001 (versión GA estable).
# ============================================================

# Modelo 1: Gemini 2.0 Flash 001 (versión estable, excelente para investigacion)
llm_investigador = LLM(
    model="openrouter/google/gemini-2.0-flash-001",
    api_key=openrouter_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.3,
    max_tokens=4000,
)

# Modelo 2: Llama 3.1 8B (redaccion y sintesis — más capaz que 3.2 3B)
llm_redactor = LLM(
    model="openrouter/meta-llama/llama-3.1-8b-instruct",
    api_key=openrouter_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7,
    max_tokens=3000,
)

# ============================================================
# VERIFICACION DE MODELOS
# crewai.LLM expone .call(messages) — mismo mecanismo que usa el agente
# ============================================================

def verificar_modelo(llm, nombre):
    """Verifica que el modelo responde correctamente usando crewai.LLM.call()"""
    try:
        response = llm.call([{"role": "user", "content": "Responde solo con: OK"}])
        preview = (response[:60] + "...") if response and len(response) > 60 else response
        print(f"{nombre} - Modelo funcionando: {preview}")
        return True
    except Exception as e:
        print(f"{nombre} - Error: {e}")
        return False

print("="*60)
print("CONFIGURANDO OPENROUTER CON MODELOS GRATUITOS")
print("="*60)
print()

print("Verificando modelos:")
verificar_modelo(llm_investigador, "Investigador (Gemini 2.0 Flash 001)")
verificar_modelo(llm_redactor, "Redactor (Llama 3.1 8B)")
print()

# ============================================================
# AGENTE 1: INVESTIGADOR (Gemini 2.0 Flash 001)
# ============================================================
investigador = Agent(
    role="Investigador Senior en IA y Nanotecnologia",
    goal=(
        "Recopilar, analizar y sintetizar literatura cientifica sobre el tema asignado. "
        "Identificar tendencias emergentes, gaps en el conocimiento y oportunidades de investigacion."
    ),
    backstory=(
        "Eres un investigador con 15 anos de experiencia en IA aplicada a nanomateriales. "
        "Has publicado en Nature Nanotechnology y ACS Nano. "
        "Tu metodologia combina revisiones sistematicas con sintesis cualitativa. "
        "Siempre citas fuentes especificas y distingues entre hallazgos replicados y prometedores pero preliminares. "
        "Tu estilo de escritura es preciso, denso en informacion y directo al punto."
        "\n\n"
        "IMPORTANTE: Basa tus respuestas en conocimiento cientifico establecido. "
        "Cuando menciones investigaciones, indica si son hallazgos replicados o resultados preliminares."
    ),
    llm=llm_investigador,
    verbose=True,
    max_iter=3,
    memory=True,
    allow_delegation=False,
)

# ============================================================
# AGENTE 2: REDACTOR (Llama 3.1 8B)
# ============================================================
redactor = Agent(
    role="Redactor Tecnico Senior",
    goal=(
        "Transformar informacion tecnica densa en documentos claros, bien estructurados y "
        "accesibles para audiencias con formacion universitaria en ciencias."
    ),
    backstory=(
        "Eres experto en comunicacion cientifica con experiencia en revistas de divulgacion como "
        "Scientific American y MIT Technology Review. "
        "Sabes convertir conceptos complejos en narrativas comprensibles sin perder rigor. "
        "Estructura tu output con headers, bullet points y ejemplos concretos. "
        "Siempre incluyes una seccion de 'Implicaciones Practicas' al final."
        "\n\n"
        "Estilo: Usa analogias accesibles, evita jerga innecesaria, "
        "y asegura que cada seccion fluya logicamente de una a la siguiente."
    ),
    llm=llm_redactor,
    verbose=True,
    max_iter=3,
    memory=True,
    allow_delegation=False,
)

# ============================================================
# INFORMACION DE LOS AGENTES
# ============================================================
print("="*60)
print("AGENTES CONFIGURADOS EXITOSAMENTE")
print("="*60)
print(f"\nInvestigador: {investigador.role}")
print(f"   Modelo: openrouter/google/gemini-2.0-flash-001")
print(f"   Temperature: 0.3 (precision)")
print(f"   Max tokens: 4000")
print(f"   Funcion: Analisis profundo y sintesis de informacion")

print(f"\nRedactor: {redactor.role}")
print(f"   Modelo: openrouter/meta-llama/llama-3.1-8b-instruct")
print(f"   Temperature: 0.7 (creatividad)")
print(f"   Max tokens: 3000")
print(f"   Funcion: Redaccion clara y estructuracion de contenido")

print("\n" + "="*60)
print("TODO LISTO - Los agentes estan preparados para trabajar")
print("="*60)


CONFIGURANDO OPENROUTER CON MODELOS GRATUITOS

Verificando modelos:
Investigador (Gemini 2.0 Flash 001) - Modelo funcionando: OK

Redactor (Llama 3.1 8B) - Modelo funcionando: OK
Provider List: https://docs.litellm.ai/docs/providers



AGENTES CONFIGURADOS EXITOSAMENTE

Investigador: Investigador Senior en IA y Nanotecnologia
   Modelo: openrouter/google/gemini-2.0-flash-001
   Temperature: 0.3 (precision)
   Max tokens: 4000
   Funcion: Analisis profundo y sintesis de informacion

Redactor: Redactor Tecnico Senior
   Modelo: openrouter/meta-llama/llama-3.1-8b-instruct
   Temperature: 0.7 (creatividad)
   Max tokens: 3000
   Funcion: Redaccion clara y estructuracion de contenido

TODO LISTO - Los agentes estan preparados para trabajar



## Tabla de Modelos Gratuitos en OpenRouter

| Modelo | Código | Mejor Para | Temperature | Max Tokens | Contexto | Velocidad |
|--------|--------|------------|-------------|------------|----------|----------|
| **Gemini 2.0 Flash** | `google/gemini-2.0-flash-exp` | Investigación, análisis, razonamiento complejo | 0.3-0.5 | 8192 | 1M tokens | Muy rápida |
| **Gemini 2.0 Flash Lite** | `google/gemini-2.0-flash-lite-preview-02-05` | Procesamiento rápido, tareas simples | 0.5 | 8192 | 1M tokens | Extremadamente rápida |
| **Llama 3.2 3B** | `meta-llama/llama-3.2-3b-instruct` | Redacción, síntesis, creatividad | 0.7 | 8192 | 128K tokens | Rápida |
| **Llama 3.1 8B** | `meta-llama/llama-3.1-8b-instruct` | Equilibrio entre calidad y velocidad | 0.5-0.7 | 8192 | 128K tokens | Buena |
| **Qwen 2.5 7B** | `qwen/qwen-2.5-7b-instruct` | Análisis técnico, código, matemáticas | 0.3 | 8192 | 128K tokens | Buena |
| **Qwen 2.5 3B** | `qwen/qwen-2.5-3b-instruct` | Tareas ligeras, embeddings | 0.5 | 8192 | 32K tokens | Rápida |
| **Phi-3.5 Mini** | `microsoft/phi-3.5-mini-128k-instruct` | Texto claro, explicaciones educativas | 0.7 | 8192 | 128K tokens | Rápida |
| **Phi-3 Mini** | `microsoft/phi-3-mini-128k-instruct` | Documentación técnica, tutoriales | 0.6 | 8192 | 128K tokens | Rápida |
| **Mistral 7B** | `mistralai/mistral-7b-instruct` | Conversación, razonamiento general | 0.5 | 8192 | 32K tokens | Buena |
| **Mixtral 8x7B** | `mistralai/mixtral-8x7b-instruct` | Razonamiento complejo, multitarea | 0.4 | 8192 | 32K tokens | Media |
| **DeepSeek V2.5** | `deepseek/deepseek-v2.5` | Análisis profundo, código, matemáticas | 0.3 | 8192 | 128K tokens | Media |
| **DeepSeek Coder** | `deepseek/deepseek-coder-6.7b-instruct` | Programación, código técnico | 0.2 | 8192 | 16K tokens | Buena |
| **Hermes 3** | `nousresearch/hermes-3-llama-3.1-8b` | Creatividad, escritura literaria | 0.8 | 8192 | 128K tokens | Buena |
| **OpenChat 3.5** | `openchat/openchat-3.5-0106` | Conversación natural, chatbots | 0.6 | 8192 | 8K tokens | Rápida |
| **Zephyr 7B** | `huggingface/zephyr-7b-beta` | Asistencia general, seguimiento instrucciones | 0.5 | 8192 | 8K tokens | Rápida |

## Combinaciones Recomendadas

### Opción 1: Investigación + Redacción (Equilibrio)
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Gemini 2.0 Flash | `google/gemini-2.0-flash-exp` | 0.3 |
| Redactor | Llama 3.2 3B | `meta-llama/llama-3.2-3b-instruct` | 0.7 |

### Opción 2: Máxima Calidad (Ambos con Gemini)
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Gemini 2.0 Flash | `google/gemini-2.0-flash-exp` | 0.3 |
| Redactor | Gemini 2.0 Flash | `google/gemini-2.0-flash-exp` | 0.7 |

### Opción 3: Especializado Técnico
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Qwen 2.5 7B | `qwen/qwen-2.5-7b-instruct` | 0.3 |
| Redactor | Phi-3.5 Mini | `microsoft/phi-3.5-mini-128k-instruct` | 0.7 |

### Opción 4: Código y Matemáticas
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | DeepSeek V2.5 | `deepseek/deepseek-v2.5` | 0.2 |
| Redactor | Qwen 2.5 7B | `qwen/qwen-2.5-7b-instruct` | 0.5 |

### Opción 5: Creatividad y Narrativa
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Mistral 7B | `mistralai/mistral-7b-instruct` | 0.4 |
| Redactor | Hermes 3 | `nousresearch/hermes-3-llama-3.1-8b` | 0.8 |

### Opción 6: Rápido y Ligero
| Rol | Modelo | Código | Temperature |
|-----|--------|--------|-------------|
| Investigador | Gemini 2.0 Flash Lite | `google/gemini-2.0-flash-lite-preview-02-05` | 0.5 |
| Redactor | Qwen 2.5 3B | `qwen/qwen-2.5-3b-instruct` | 0.6 |

## Código de Ejemplo para Cambiar Combinaciones

```python
# Configuracion base para todos los modelos
def crear_llm(modelo, temperatura, max_tokens=4000):
    return ChatOpenAI(
        model=modelo,
        openai_api_key=os.getenv("OPENROUTER_API_KEY"),
        temperature=temperatura,
        max_tokens=max_tokens,
        base_url="https://openrouter.ai/api/v1"
    )

# Opcion 1: Investigacion + Redaccion
llm_investigador = crear_llm("google/gemini-2.0-flash-exp", 0.3)
llm_redactor = crear_llm("meta-llama/llama-3.2-3b-instruct", 0.7)

# Opcion 2: Ambos Gemini
llm_investigador = crear_llm("google/gemini-2.0-flash-exp", 0.3)
llm_redactor = crear_llm("google/gemini-2.0-flash-exp", 0.7)

# Opcion 3: Especializado tecnico
llm_investigador = crear_llm("qwen/qwen-2.5-7b-instruct", 0.3)
llm_redactor = crear_llm("microsoft/phi-3.5-mini-128k-instruct", 0.7)

# Opcion 4: Codigo y matematicas
llm_investigador = crear_llm("deepseek/deepseek-v2.5", 0.2)
llm_redactor = crear_llm("qwen/qwen-2.5-7b-instruct", 0.5)

# Opcion 5: Creatividad
llm_investigador = crear_llm("mistralai/mistral-7b-instruct", 0.4)
llm_redactor = crear_llm("nousresearch/hermes-3-llama-3.1-8b", 0.8)

# Opcion 6: Rapido y ligero
llm_investigador = crear_llm("google/gemini-2.0-flash-lite-preview-02-05", 0.5, 2000)
llm_redactor = crear_llm("qwen/qwen-2.5-3b-instruct", 0.6, 2000)
```

## Recomendaciones por Tipo de Tarea

| Tarea | Modelo Recomendado | Temperature |
|-------|-------------------|-------------|
| Investigación científica | Gemini 2.0 Flash | 0.3 |
| Análisis de código | DeepSeek V2.5 | 0.2 |
| Redacción técnica | Phi-3.5 Mini | 0.7 |
| Escritura creativa | Hermes 3 | 0.8 |
| Conversación | Mistral 7B | 0.6 |
| Procesamiento rápido | Gemini Flash Lite | 0.5 |
| Matemáticas | Qwen 2.5 7B | 0.3 |
| Documentación | Phi-3 Mini | 0.6 |


```python

# Opción 1: Ambos con Gemini (muy capaz)
llm_gemini = ChatOpenAI(
    model="google/gemini-2.0-flash-exp",
    openrouter_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.5,
    max_tokens=4000,
    base_url="https://openrouter.ai/api/v1"
)

# Opción 2: Investigador con Qwen 2.5 (bueno para análisis técnico)
llm_qwen = ChatOpenAI(
    model="qwen/qwen-2.5-7b-instruct",
    openrouter_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.3,
    max_tokens=4000,
    base_url="https://openrouter.ai/api/v1"
)

# Opción 3: Redactor con Phi-3.5 (bueno para texto claro)
llm_phi = ChatOpenAI(
    model="microsoft/phi-3.5-mini-128k-instruct",
    openrouter_api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0.7,
    max_tokens=3000,
    base_url="https://openrouter.ai/api/v1"
)

```


Todos estos modelos son **completamente gratuitos** en OpenRouter y no requieren créditos ni pagos.

### 3.2 Task — description, expected_output

Una `Task` define **qué hacer** (description) y **qué formato de output se espera** (expected_output). El campo `expected_output` actúa como el criterio de evaluación interno del agente — si el output no lo cumple, el agente intenta de nuevo.

Las tasks pueden tener **context**: lista de otras tasks cuyo output se pasa como entrada. Esto crea dependencias explícitas entre tareas.

In [10]:
# Paso 3: Definir tasks con contexto entre ellas

tarea_investigacion = Task(
    description=(
        "Realiza una revisión del estado del arte sobre el uso de IA (especialmente sistemas multi-agente) "
        "en la investigación de nanomateriales. "
        "Cubre: (1) técnicas de ML para predicción de propiedades, "
        "(2) optimización de síntesis con RL, "
        "(3) análisis automatizado de imágenes TEM/SEM, "
        "(4) agentes para revisión de literatura."
    ),
    expected_output=(
        "Un informe estructurado de 400-600 palabras con: "
        "(1) resumen ejecutivo de 3-4 oraciones, "
        "(2) estado del arte en 4 subsecciones con las áreas indicadas, "
        "(3) identificación de 2-3 gaps de investigación, "
        "(4) al menos 5 referencias específicas (autor, año, tema)."
    ),
    agent=investigador,
)

tarea_redaccion = Task(
    description=(
        "Basándote en el informe de revisión del investigador, "
        "crea un artículo de divulgación técnica de 300-400 palabras "
        "que explique el estado del arte a una audiencia de ingenieros de materiales "
        "sin experiencia en IA. "
        "Incluye: título atractivo, introducción, 3 secciones temáticas, implicaciones prácticas."
    ),
    expected_output=(
        "Un artículo de divulgación con título, 4 secciones claramente marcadas con headers, "
        "sin jerga de IA innecesaria, con al menos un ejemplo concreto en cada sección, "
        "y una sección final 'Implicaciones Prácticas' de 3+ bullet points."
    ),
    agent=redactor,
    context=[tarea_investigacion],  # el output de investigación es input aquí
)

print("Tasks definidas:")
print(f"  - tarea_investigacion (agente: {tarea_investigacion.agent.role})")
print(f"  - tarea_redaccion (agente: {tarea_redaccion.agent.role}, contexto: tarea_investigacion)")

Tasks definidas:
  - tarea_investigacion (agente: Investigador Senior en IA y Nanotecnologia)
  - tarea_redaccion (agente: Redactor Tecnico Senior, contexto: tarea_investigacion)


### 3.3 Crew y Process: Sequential vs Hierarchical

In [11]:
# ────────────────────────────────────────────────────────────
# Output esperado: crew secuencial investigador → redactor
# ────────────────────────────────────────────────────────────

# Paso 4: Crear la Crew con Process.sequential
# Sequential = las tasks se ejecutan en el orden en que se definieron
crew_secuencial = Crew(
    agents=[investigador, redactor],
    tasks=[tarea_investigacion, tarea_redaccion],
    process=Process.sequential,
    verbose=True,
)

print("Crew secuencial configurada:")
print(f"  Agentes: {len(crew_secuencial.agents)}")
print(f"  Tasks: {len(crew_secuencial.tasks)}")
print(f"  Proceso: Sequential")

Crew secuencial configurada:
  Agentes: 2
  Tasks: 2
  Proceso: Sequential


In [12]:
# Paso 5: Ejecutar la Crew
# NOTA: Esta celda hace llamadas reales al LLM — puede tardar 30-90 segundos
print("Iniciando misión de investigación...")
print("=" * 60)

resultado_secuencial = crew_secuencial.kickoff()

print("\n" + "=" * 60)
print("RESULTADO FINAL DE LA CREW:")
print("=" * 60)
print(resultado_secuencial.raw)

Iniciando misión de investigación...
# Agent: Investigador Senior en IA y Nanotecnologia
## Task: Realiza una revisión del estado del arte sobre el uso de IA (especialmente sistemas multi-agente) en la investigación de nanomateriales. Cubre: (1) técnicas de ML para predicción de propiedades, (2) optimización de síntesis con RL, (3) análisis automatizado de imágenes TEM/SEM, (4) agentes para revisión de literatura.


# Agent: Investigador Senior en IA y Nanotecnologia
## Final Answer: 
**Estado del Arte: IA y Sistemas Multi-Agente en la Investigación de Nanomateriales**

**Resumen Ejecutivo:** La aplicación de inteligencia artificial (IA) está transformando la investigación de nanomateriales, acelerando el descubrimiento y optimización de nuevos materiales con propiedades mejoradas. Técnicas de aprendizaje automático (ML) predicen propiedades con alta precisión, el aprendizaje por refuerzo (RL) optimiza las rutas de síntesis, el análisis automatizado de imágenes de microscopía electróni

**Output esperado:** Primero verás el `investigador` produciendo el informe de revisión (con `Thought:` y contenido). Luego el `redactor` toma ese informe como contexto y produce el artículo de divulgación. Nótese que el redactor tiene acceso al output del investigador automáticamente via el campo `context=[tarea_investigacion]`.

**Interpretación:** El parámetro `memory=True` en el agente investigador hace que recuerde el contexto de su tarea cuando el redactor le pide aclaraciones. La `backstory` impacta directamente en el tono y el nivel de detalle del output — el investigador cita referencias específicas, el redactor incluye la sección de implicaciones prácticas, tal como sus backstories les instruyen.

---
## 4. Tools en CrewAI <a id='4-tools'></a>

### 4.1 Tools Nativas de CrewAI

CrewAI incluye tools pre-construidas como `SerperDevTool` (búsqueda web), `FileReadTool`, `CSVSearchTool`, entre otras. Requieren sus respectivas API keys.

### 4.2 Bridging: Tools de LangChain en CrewAI

Una de las ventajas clave de CrewAI es que acepta tools definidas con el decorador `@tool` de LangChain directamente.

In [14]:
# ────────────────────────────────────────────────────────────
# Output esperado: agente CrewAI usando tools nativas de CrewAI
# ────────────────────────────────────────────────────────────
import math
from typing import Type
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

# Nota: CrewAI 0.80.0 requiere crewai.tools.BaseTool — no acepta
# langchain_core StructuredTool directamente en Agent.tools.

# ============================================================
# Paso 1: Schemas de input (requerido por BaseTool)
# ============================================================

class BuscarNanoInput(BaseModel):
    material: str = Field(description="Nombre del material (ej: 'oro', 'grafeno', 'plata')")

class EnergiaInput(BaseModel):
    radio_nm: float = Field(description="Radio de la nanopartícula en nanómetros")
    material: str = Field(description="Nombre del material (ej: 'oro', 'dioxido de titanio')")

# ============================================================
# Paso 2: Definir tools heredando de crewai.tools.BaseTool
# ============================================================

class BuscarDatosNanoparticulas(BaseTool):
    name: str = "buscar_datos_nanoparticulas"
    description: str = (
        "Busca propiedades físico-químicas de una nanopartícula o material dado. "
        "Útil cuando necesitas datos sobre tamaño típico, propiedades ópticas o magnéticas. "
        "Input: nombre del material (ej: 'oro', 'dióxido de titanio', 'grafeno')"
    )
    args_schema: Type[BaseModel] = BuscarNanoInput

    def _run(self, material: str) -> str:
        db = {
            "oro": {
                "tamaño_tipico": "1-100 nm",
                "propiedades": "SPR en ~520nm, biocompatible, alta estabilidad química",
                "aplicaciones": "biomedicina, catálisis, sensores",
            },
            "dioxido de titanio": {
                "tamaño_tipico": "5-50 nm",
                "propiedades": "fotocatálisis UV, blanco intenso, alta refracción",
                "aplicaciones": "protector solar, fotovoltaica, pinturas",
            },
            "grafeno": {
                "tamaño_tipico": "monocapa ~0.3nm",
                "propiedades": "conductividad eléctrica 1M S/m, resistencia mecánica 130 GPa",
                "aplicaciones": "electrónica, compuestos, baterías",
            },
            "plata": {
                "tamaño_tipico": "5-100 nm",
                "propiedades": "antibacteriano, SERS, SPR en ~400nm",
                "aplicaciones": "antimicrobiano, diagnóstico, tintas conductoras",
            },
        }
        m = material.lower()
        for key, data in db.items():
            if key in m or m in key:
                return (
                    f"Material: {key}\n"
                    f"Tamaño: {data['tamaño_tipico']}\n"
                    f"Propiedades: {data['propiedades']}\n"
                    f"Aplicaciones: {data['aplicaciones']}"
                )
        return f"Material '{material}' no encontrado. Disponibles: {', '.join(db.keys())}"


class CalcularEnergiaSurface(BaseTool):
    name: str = "calcular_energia_surface"
    description: str = (
        "Calcula la energía de superficie aproximada de una nanopartícula esférica. "
        "Aplica la aproximación: E_surf = 4πr² * γ donde γ es la energía de superficie específica. "
        "Inputs: radio_nm (radio en nanómetros), material (nombre del material)"
    )
    args_schema: Type[BaseModel] = EnergiaInput

    def _run(self, radio_nm: float, material: str) -> str:
        gamma_db = {"oro": 1.50, "plata": 1.20, "dioxido de titanio": 0.90, "grafeno": 0.05}
        gamma = gamma_db.get(material.lower(), 1.0)
        radio_m = radio_nm * 1e-9
        area = 4 * math.pi * radio_m ** 2
        energia = area * gamma
        return (
            f"Radio: {radio_nm} nm = {radio_m:.2e} m\n"
            f"Área superficial: {area:.4e} m²\n"
            f"γ ({material}): {gamma} J/m²\n"
            f"Energía de superficie: {energia:.4e} J"
        )


# ============================================================
# Paso 3: Instanciar las tools e inyectarlas en el agente
# ============================================================
buscar_tool = BuscarDatosNanoparticulas()
energia_tool = CalcularEnergiaSurface()

analista_nano = Agent(
    role="Analista de Nanomateriales",
    goal="Analizar propiedades físico-químicas de nanomateriales y realizar cálculos de energía de superficie.",
    backstory=(
        "Especialista en caracterización de nanomateriales con experiencia en AFM, TEM y DLS. "
        "Rigoroso en los cálculos y siempre presenta resultados con unidades correctas y contexto físico."
    ),
    tools=[buscar_tool, energia_tool],
    llm=llm,
    verbose=True,
    max_iter=4,
)

tarea_analisis = Task(
    description=(
        "Análisis de nanopartículas de oro:\n"
        "1. Busca las propiedades del oro nanométrico\n"
        "2. Calcula la energía de superficie para radios de 5nm, 10nm y 20nm\n"
        "3. Explica cómo cambia la energía de superficie con el tamaño y por qué esto importa"
    ),
    expected_output=(
        "Un análisis que incluya: propiedades del nanoro obtenidas de la base de datos, "
        "tabla comparativa de energías de superficie para 3 tamaños, "
        "interpretación física del efecto de escala en al menos 2 párrafos."
    ),
    agent=analista_nano,
)

crew_analista = Crew(
    agents=[analista_nano],
    tasks=[tarea_analisis],
    process=Process.sequential,
    verbose=True,
)

print("Tools CrewAI definidas:")
print(f"  - {buscar_tool.name}")
print(f"  - {energia_tool.name}")
print("Crew de analista configurada.")


Tools CrewAI definidas:
  - buscar_datos_nanoparticulas
  - calcular_energia_surface
Crew de analista configurada.


In [15]:
# Ejecutar el análisis
resultado_analisis = crew_analista.kickoff()
print("\n" + "=" * 60)
print("ANÁLISIS DE NANOPARTÍCULAS:")
print(resultado_analisis.raw)

# Agent: Analista de Nanomateriales
## Task: Análisis de nanopartículas de oro:
1. Busca las propiedades del oro nanométrico
2. Calcula la energía de superficie para radios de 5nm, 10nm y 20nm
3. Explica cómo cambia la energía de superficie con el tamaño y por qué esto importa

Provider List: https://docs.litellm.ai/docs/providers



# Agent: Analista de Nanomateriales
## Using tool: buscar_datos_nanoparticulas
## Tool Input: 
"{\"material\": \"oro\"}"
## Tool Output: 
Material: oro
Tamaño: 1-100 nm
Propiedades: SPR en ~520nm, biocompatible, alta estabilidad química
Aplicaciones: biomedicina, catálisis, sensores


# Agent: Analista de Nanomateriales
## Using tool: calcular_energia_surface
## Tool Input: 
"{\"radio_nm\": 5, \"material\": \"oro\"}"
## Tool Output: 
Radio: 5.0 nm = 5.00e-09 m
Área superficial: 3.1416e-16 m²
γ (oro): 1.5 J/m²
Energía de superficie: 4.7124e-16 J


# Agent: Analista de Nanomateriales
## Using tool: calcular_energia_surface
## Tool Input: 
"{\"radio_nm\": 10,

---
## 5. Process Hierarchical: Manager + Delegates <a id='5-hierarchical'></a>

En `Process.sequential` las tareas se ejecutan en orden fijo. En `Process.hierarchical`, un **agente manager** decide dinámicamente qué agente realiza cada tarea y en qué orden, basándose en las instrucciones y el estado del proyecto.

```
SEQUENTIAL:        Manager  (solo orquesta)
  tarea1 → a1     /     \
  tarea2 → a2    a1     a2
  tarea3 → a3    |      |
                t1     t2/t3 (decide el manager)

HIERARCHICAL:
  Manager decide: "a1 hace t1, luego a2 decide si hace t2 o t3 según el resultado de t1"
```

In [17]:

# ────────────────────────────────────────────────────────────
# Output esperado: crew jerárquica con manager dinámico
# ────────────────────────────────────────────────────────────

# Paso 1: Definir el agente manager
# Con Process.hierarchical, CrewAI crea un manager interno
# O podemos definir uno explícito con manager_agent

director_proyecto = Agent(
    role="Director de Proyecto de Investigación",
    goal=(
        "Coordinar al equipo de investigación para producir un informe técnico de alta calidad. "
        "Delegar tareas a los especialistas adecuados y asegurarse de que cada output cumple los estándares."
    ),
    backstory=(
        "Eres un director con 20 años de experiencia liderando proyectos de I+D en nanotecnología. "
        "Sabes identificar qué especialista es el más adecuado para cada tarea. "
        "Eres exigente con la calidad pero eficiente — no pides trabajo innecesario. "
        "Cuando el output de un especialista no cumple los criterios, lo devuelves con feedback específico."
    ),
    llm=llm,
    allow_delegation=True,  # puede delegar a otros agentes
    verbose=True,
)

# Paso 2: Crew jerárquica
# Task única de alto nivel — el manager la descompone y delega
tarea_proyecto = Task(
    description=(
        "Producir un informe técnico completo sobre 'Aplicaciones de IA en síntesis de nanopartículas'. "
        "El informe debe incluir: revisión del estado del arte, análisis de propiedades de al menos 2 materiales, "
        "y una sección de divulgación accesible para no-especialistas."
    ),
    expected_output=(
        "Informe estructurado con: portada, índice implícito, sección técnica (800+ palabras), "
        "análisis de materiales con datos numéricos, y sección de divulgación (300+ palabras). "
        "Total: 1100+ palabras con nivel de calidad publicable."
    ),
    agent=director_proyecto,
)

# NOTA CrewAI 0.80.0: el manager_agent NO debe incluirse en agents[]
# El manager orquesta pero no es un worker — va solo en manager_agent=
crew_jerarquica = Crew(
    agents=[investigador, analista_nano, redactor],  # solo workers
    tasks=[tarea_proyecto],
    process=Process.hierarchical,
    manager_agent=director_proyecto,
    verbose=True,
)

print("Crew jerárquica configurada:")
print(f"  Manager: {director_proyecto.role}")
print(f"  Equipo: {[a.role for a in [investigador, analista_nano, redactor]]}")


Crew jerárquica configurada:
  Manager: Director de Proyecto de Investigación
  Equipo: ['Investigador Senior en IA y Nanotecnologia', 'Analista de Nanomateriales', 'Redactor Tecnico Senior']


In [18]:
# NOTA: Process.hierarchical hace más llamadas al LLM que Sequential
# Es más lento pero produce outputs de mayor calidad en tareas complejas
# Tiempo estimado: 2-4 minutos con Gemini Flash

print("Iniciando proyecto con crew jerárquica...")
print("El director asignará subtareas dinámicamente a los especialistas.")
print("=" * 60)

resultado_jerarquico = crew_jerarquica.kickoff()

print("\n" + "=" * 60)
print("INFORME FINAL DEL DIRECTOR:")
print("=" * 60)
print(resultado_jerarquico.raw[:1000], "...[continúa]")

Iniciando proyecto con crew jerárquica...
El director asignará subtareas dinámicamente a los especialistas.
# Agent: Director de Proyecto de Investigación
## Task: Producir un informe técnico completo sobre 'Aplicaciones de IA en síntesis de nanopartículas'. El informe debe incluir: revisión del estado del arte, análisis de propiedades de al menos 2 materiales, y una sección de divulgación accesible para no-especialistas.


# Agent: Director de Proyecto de Investigación
## Using tool: Delegate work to coworker
## Tool Input: 
"{\"task\": \"Realizar una revisi\\u00f3n exhaustiva del estado del arte sobre las aplicaciones de la Inteligencia Artificial en la s\\u00edntesis de nanopart\\u00edculas. Identificar al menos dos materiales relevantes (por ejemplo, nanopart\\u00edculas de oro, plata, \\u00f3xidos met\\u00e1licos, puntos cu\\u00e1nticos, etc.) y analizar sus propiedades clave, incluyendo datos num\\u00e9ricos cuando sea posible, en el contexto de su s\\u00edntesis asistida por IA.

**Interpretación:** En `Process.hierarchical`, el director decide en tiempo de ejecución si delega al investigador primero, luego al analista, luego al redactor — o si los llama en un orden diferente según los resultados intermedios. Esto no es posible en `Process.sequential` donde el orden es fijo. La diferencia es visible en los logs: con hierarchical verás al manager evaluando outputs y tomando decisiones de delegación explícitas.

---
## 6. Skill: Output Scorer — Evaluación Automática <a id='6-output-scorer'></a>

La skill `output_scorer` cierra el ciclo de calidad: permite evaluar automáticamente el output de cualquier agente o crew usando criterios definidos programáticamente.

In [23]:
# ────────────────────────────────────────────────────────────
# Output esperado: evaluar el informe generado por la crew
# ────────────────────────────────────────────────────────────

# Paso 1: Cargar la skill (con reload para limpiar caché del kernel)
import importlib
import sys

mods_to_reload = [m for k, m in sys.modules.items() if "output_scorer" in k]
for mod in mods_to_reload:
    importlib.reload(mod)

from external_skills.evaluation.output_scorer import (
    score_with_llm, score_heuristic, DEFAULT_CRITERIA, ScoreResult, SKILL_METADATA as OS_META
)

print(f"Skill cargada: {OS_META['name']} v{OS_META['version']}")
print(f"Criterios por defecto: {list(DEFAULT_CRITERIA.keys())}")


Skill cargada: output_scorer v1.1.0
Criterios por defecto: ['length', 'structure', 'specificity', 'actionability']


In [24]:
# Paso 2: Evaluación heurística del informe (sin costo de LLM)
score_heuristic_result = score_heuristic(
    output=resultado_secuencial.raw,
    task_description="Artículo de divulgación técnica sobre IA en nanotecnología"
)

print("Evaluación Heurística:")
print(f"  Score: {score_heuristic_result.score:.2f}/{score_heuristic_result.max_score}")
print(f"  Dimensiones:")
for dim, val in score_heuristic_result.breakdown.items():
    print(f"    {dim}: {val:.2f}")
print(f"  Feedback: {score_heuristic_result.feedback[:200]}")

Evaluación Heurística:
  Score: 0.46/1.0
  Dimensiones:
    length: 0.75
    structure: 0.50
    specificity: 0.60
    actionability: 0.00
  Feedback: Evaluación heurística de 'Artículo de divulgación técnica sobre IA en nanotecnología': length: 75%, structure: 50%, specificity: 60%, actionability: 0%. Score total: 46%.


In [25]:
# Paso 3: Evaluación con LLM (más detallada)
# Criterios personalizados para este tipo de output
criterios_articulo = {
    "rigor_tecnico": 0.30,      # precisión de contenido técnico
    "claridad": 0.25,           # accesibilidad para no-especialistas
    "estructura": 0.20,         # organización y formato
    "completitud": 0.25,        # cubre todas las áreas requeridas
}

print("Iniciando evaluación con LLM (puede tardar 5-10 segundos)...")
score_llm_result = score_with_llm(
    output=resultado_secuencial.raw,
    task_description="Artículo de divulgación sobre IA en nanotecnología para ingenieros de materiales",
    criteria=criterios_articulo,
    llm=llm
)

print("\nEvaluación con LLM:")
print(f"  Score total: {score_llm_result.score:.2f}/{score_llm_result.max_score}")
print(f"  Dimensiones:")
for dim, val in score_llm_result.breakdown.items():
    bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
    print(f"    {dim:<20} {bar} {val:.2f}")
print(f"\n  Feedback completo:")
print(f"  {score_llm_result.feedback}")

Iniciando evaluación con LLM (puede tardar 5-10 segundos)...

Evaluación con LLM:
  Score total: 0.85/1.0
  Dimensiones:
    rigor_tecnico        █████████░ 0.90
    claridad             █████████░ 0.90
    estructura           █████████░ 0.90
    completitud          ███████░░░ 0.70

  Feedback completo:
  Evaluación LLM de 'Artículo de divulgación sobre IA en nanotecnología para inge': rigor_tecnico: 90%, claridad: 90%, estructura: 90%, completitud: 70%. Score total: 85%.


**Interpretación:** Tener scoring automático transforma la evaluación de agentes de una actividad subjetiva en una métrica medible. Con `score_heuristic` podemos verificar rápidamente si el output cumple requisitos básicos (longitud, estructura, keywords). Con `score_with_llm` obtenemos una evaluación semántica más rica. En un pipeline de producción, esta skill se usaría como nodo de quality gate en el grafo LangGraph — si el score es menor a 0.7, el output se rechaza y se regenera.

---
## 7. Proyecto: Equipo de Investigación en Nanotecnología <a id='7-proyecto'></a>

Integración completa: crew de 4 agentes especializados + tools de dominio + evaluación automática con output_scorer.

In [29]:
# ============================================================
# PROYECTO: Equipo Completo de Investigación en Nanomateriales
# ============================================================

# Paso 1: Definir el equipo completo

revisor_calidad = Agent(
    role="Revisor de Calidad Científica",
    goal=(
        "Verificar que el informe final cumple estándares de rigor científico, "
        "consistencia interna, y precisión técnica. "
        "Identificar afirmaciones sin respaldo y recomendar correcciones."
    ),
    backstory=(
        "Ex-editor de ACS Nano con 10 años de experiencia en peer review. "
        "Tienes ojo clínico para detectar over-claims, datos incorrectos y falta de rigor. "
        "Tu review es siempre constructivo: señalas el problema Y propones la corrección específica. "
        "No apruebas un documento si contiene errores factuales o afirmaciones sin evidencia."
    ),
    llm=llm,
    verbose=True,
    max_iter=2,
)

# Paso 2: Pipeline de tareas con dependencias en cadena
# investigacion → analisis → redaccion → revision

t1_investigacion = Task(
    description=(
        "Investiga el estado actual de la síntesis de nanopartículas de oro usando métodos asistidos por IA. "
        "Enfócate en: (1) optimización por reinforcement learning, "
        "(2) predicción de propiedades por ML, (3) control de calidad automatizado. "
        "Incluye 3+ referencias específicas."
    ),
    expected_output="Informe técnico de 300-400 palabras con 3 subsecciones temáticas y 3+ referencias.",
    agent=investigador,
)

t2_analisis = Task(
    description=(
        "Basándote en el informe de investigación, "
        "realiza un análisis cuantitativo de las propiedades de las nanopartículas de oro. "
        "Calcula la energía de superficie para radios de 2nm, 5nm y 10nm. "
        "Busca las propiedades en la base de datos y presenta los resultados en formato de tabla."
    ),
    expected_output=(
        "Análisis cuantitativo con: propiedades del nanoro de la base de datos, "
        "tabla de energías para 3 tamaños, interpretación física en 1 párrafo."
    ),
    agent=analista_nano,
    context=[t1_investigacion],
)

t3_redaccion = Task(
    description=(
        "Integra el informe de investigación y el análisis cuantitativo en un "
        "documento técnico coherente de 600-800 palabras. "
        "Audiencia: ingenieros de producción en una empresa de nanomateriales. "
        "Incluye: resumen ejecutivo (5 líneas), cuerpo técnico, tabla de datos, conclusiones accionables."
    ),
    expected_output=(
        "Documento integrado con resumen ejecutivo, 3 secciones técnicas, "
        "al menos 1 tabla de datos, y 3+ conclusiones accionables en bullet points."
    ),
    agent=redactor,
    context=[t1_investigacion, t2_analisis],
)

t4_revision = Task(
    description=(
        "Revisa el documento integrado en busca de: inconsistencias entre secciones, "
        "afirmaciones sin respaldo, errores en los cálculos de energía, y problemas de claridad. "
        "Proporciona el documento corregido o una lista de correcciones específicas."
    ),
    expected_output=(
        "Lista numerada de encontrados (o 'APROBADO' si no hay problemas) + "
        "la versión final corregida del documento completo."
    ),
    agent=revisor_calidad,
    context=[t3_redaccion],
)

# Paso 3: Crew del proyecto completo
crew_investigacion = Crew(
    agents=[investigador, analista_nano, redactor, revisor_calidad],
    tasks=[t1_investigacion, t2_analisis, t3_redaccion, t4_revision],
    process=Process.sequential,
    verbose=True,
)

# Mapa explícito de LLMs por agente
_llm_labels = {
    investigador.role:    "openrouter/google/gemini-2.0-flash-001",
    analista_nano.role:   llm.model,
    redactor.role:        "openrouter/meta-llama/llama-3.1-8b-instruct",
    revisor_calidad.role: llm.model,
}

print("Equipo completo de investigación configurado:")
print(f"  {'Agente':<40} {'LLM'}")
print(f"  {'-'*40} {'-'*40}")
for a in crew_investigacion.agents:
    print(f"  {a.role:<40} {_llm_labels[a.role]}")
print(f"\nPipeline: t1 Investigación → t2 Análisis → t3 Redacción → t4 Revisión")


Equipo completo de investigación configurado:
  Agente                                   LLM
  ---------------------------------------- ----------------------------------------
  Investigador Senior en IA y Nanotecnologia openrouter/google/gemini-2.0-flash-001
  Analista de Nanomateriales               openrouter/google/gemini-2.5-flash
  Redactor Tecnico Senior                  openrouter/meta-llama/llama-3.1-8b-instruct
  Revisor de Calidad Científica            openrouter/google/gemini-2.5-flash

Pipeline: t1 Investigación → t2 Análisis → t3 Redacción → t4 Revisión


In [27]:
# Paso 4: Ejecutar el proyecto completo
# NOTA: Esto hace 4 llamadas encadenadas al LLM — puede tardar 3-5 minutos
print("Ejecutando proyecto de investigación completo...")
print("Pipeline: Investigación → Análisis → Redacción → Revisión")
print("=" * 60)

resultado_proyecto = crew_investigacion.kickoff()

print("\n" + "=" * 60)
print("DOCUMENTO FINAL REVISADO:")
print("=" * 60)
print(resultado_proyecto.raw)

Ejecutando proyecto de investigación completo...
Pipeline: Investigación → Análisis → Redacción → Revisión
# Agent: Investigador Senior en IA y Nanotecnologia
## Task: Investiga el estado actual de la síntesis de nanopartículas de oro usando métodos asistidos por IA. Enfócate en: (1) optimización por reinforcement learning, (2) predicción de propiedades por ML, (3) control de calidad automatizado. Incluye 3+ referencias específicas.


# Agent: Investigador Senior en IA y Nanotecnologia
## Final Answer: 
**Estado Actual de la Síntesis de Nanopartículas de Oro Asistida por IA**

La síntesis de nanopartículas de oro (AuNPs) se beneficia cada vez más de la inteligencia artificial (IA), impulsando avances en optimización, predicción de propiedades y control de calidad.

**(1) Optimización por Reinforcement Learning (RL)**

El Reinforcement Learning (RL) emerge como una herramienta poderosa para optimizar las condiciones de síntesis de AuNPs.  Enfoques basados en RL permiten explorar de form

In [28]:
# Paso 5: Evaluar el output final con output_scorer
evaluacion_final = score_with_llm(
    output=resultado_proyecto.raw,
    task_description=(
        "Documento técnico sobre síntesis de nanopartículas asistida por IA, "
        "para ingenieros de producción, con datos cuantitativos y conclusiones accionables"
    ),
    criteria={
        "rigor_tecnico": 0.25,
        "datos_cuantitativos": 0.25,
        "claridad_audiencia": 0.20,
        "accionabilidad": 0.20,
        "coherencia_interna": 0.10,
    },
    llm=llm
)

print("\nEvaluación del Proyecto:")
print(f"Score: {evaluacion_final.score:.2f}/{evaluacion_final.max_score} ",
      end="")
pct = evaluacion_final.score / evaluacion_final.max_score
grade = "EXCELENTE" if pct > 0.85 else "BUENO" if pct > 0.70 else "ACEPTABLE" if pct > 0.55 else "INSUFICIENTE"
print(f"({grade})")
print()
print("Breakdown:")
for dim, val in evaluacion_final.breakdown.items():
    bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
    print(f"  {dim:<25} {bar} {val:.2f}")
print(f"\nFeedback:\n{evaluacion_final.feedback}")


Evaluación del Proyecto:
Score: 0.64/1.0 (ACEPTABLE)

Breakdown:
  rigor_tecnico             ███████░░░ 0.70
  datos_cuantitativos       ████░░░░░░ 0.40
  claridad_audiencia        ████████░░ 0.80
  accionabilidad            ██████░░░░ 0.60
  coherencia_interna        ████████░░ 0.80

Feedback:
Evaluación LLM de 'Documento técnico sobre síntesis de nanopartículas asistida ': rigor_tecnico: 70%, datos_cuantitativos: 40%, claridad_audiencia: 80%, accionabilidad: 60%, coherencia_interna: 80%. Score total: 64%.


---
## 8. Resumen y Criterios de Evaluación <a id='8-resumen'></a>

### Conceptos Clave de esta Notebook

| Concepto | Definición | Implementado en |
|----------|------------|------------------|
| **Agent** | Entidad con role, goal y backstory que define su expertise y comportamiento | Sección 3.1 |
| **Task** | Unidad de trabajo con descripción, expected_output y contexto de tareas previas | Sección 3.2 |
| **Process.sequential** | Tasks ejecutadas en orden prefijado | Sección 3.3 |
| **Process.hierarchical** | Manager asigna tasks dinámicamente según resultados | Sección 5 |
| **Tool bridging** | Tools `@tool` de LangChain funcionan directamente en CrewAI | Sección 4.2 |
| **context=[]** | Pasa el output de una task como input de otra (dependencia explícita) | Secciones 3.2, 7 |
| **Output Scorer** | Skill para evaluación automática de calidad de outputs multi-agente | Sección 6 |

### Criterios de Evaluación

- [ ] Definir 2+ agentes con role, goal y backstory diferenciados y verificar el impacto en los outputs
- [ ] Construir un pipeline de 3+ tasks con dependencias `context=[...]`
- [ ] Ejecutar el mismo pipeline con `Process.sequential` y describir la diferencia con hierarchical
- [ ] Usar `output_scorer` para evaluación automática y obtener un score > 0.7 ajustando los prompts
- [ ] Demostrar tool bridging: una tool LangChain usada por un agente CrewAI

### Ejercicio de Extensión

Construye una crew de 4 agentes para producir un plan de negocio sobre comercialización de un nanomaterial:
1. **Científico** — viable técnicamente (con tools de cálculo de propiedades)
2. **Economista** — análisis de mercado y costos
3. **Regulatorio** — aspectos legales y de seguridad nanotecnológica
4. **Redactor Ejecutivo** — integra todo en un executive summary de 1 página

Usa `output_scorer` con criterios personalizados para el dominio de negocio y ajusta las backstories hasta obtener un score > 0.85.

### Próxima Notebook

En **U5_04 — Google ADK y Protocolo A2A** exploraremos el framework oficial de Google para agentes, el protocolo Agent-to-Agent (A2A) para comunicación entre agentes de diferentes frameworks, y la integración con Gemini como LLM principal.

---
*Notebook generada siguiendo el PROTOCOLO_UNIDAD_5_MULTIAGENTE.md v1.1.0*  
*Antigravity-Nano-Research — Unit 05 Multi-Agent Systems — Marzo 2026*

## Ejercicio de Extensión — CrewAI: Plan de Negocio para Comercialización de Nanomateriales

Se construye una `Crew` de 4 agentes especializados:
1. **Científico Jefe** — Analiza viabilidad técnica usando Tools de propiedades.
2. **Economista** — Determina costos y oportunidades de mercado.
3. **Experto Regulatorio** — Evalúa riesgos de ecotoxicidad (REACH/EPA).
4. **Redactor Ejecutivo** — Sintetiza todo en un *Executive Summary* para inversores.

Incluye además un `output_scorer` basado en LLM que califica la salida final (buscando score > 0.85).

In [ ]:
import os
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from textwrap import dedent

# Inicializar LLM (gemini-1.5-flash para rapidez)
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)

# ==============================================================================
# 1. Definición de Herramientas (Tools) 
# ==============================================================================
@tool("Calculadora de Propiedades")
def calcular_propiedades(nanomaterial: str) -> str:
    """Calcula propiedades fisicoquímicas estimadas de un nanomaterial."""
    if "plata" in nanomaterial.lower() or "agnp" in nanomaterial.lower():
        return (
            "Material: Nanopartículas de Plata (AgNPs) | "
            "Propiedad principal: Antimicrobiano de amplio espectro | "
            "Toxicidad acuática: Alta (Riesgo ecotoxicológico) | "
            "Costo síntesis: Medio-Alto"
        )
    return "Material Desconocido | Viabilidad: Pendiente de validación."

# ==============================================================================
# 2. Definición de Agentes (con backstories ajustados para alto score)
# ==============================================================================
cientifico = Agent(
    role="Científico Jefe de Materiales",
    goal="Validar la viabilidad técnica y extraer propiedades fisicoquímicas del material.",
    backstory=dedent("""
        Tienes un Ph.D. en Nanotecnología con 15 años de experiencia.
        Eres increíblemente preciso y te basas en datos científicos puros. 
        Analizas si el material es técnicamente escalable e informas de las propiedades esenciales.
    """),
    tools=[calcular_propiedades],
    verbose=True,
    allow_delegation=False,
    llm=llm
)

economista = Agent(
    role="Analista Económico y de Mercado",
    goal="Evaluar costos de producción, rentabilidad y demanda potencial en el mercado.",
    backstory=dedent("""
        Eres un gurú financiero especializado en capital de riesgo (VC) de Deep Tech.
        Buscas rentabilidad (ROI) y modelos de negocio sostenibles. 
        Evalúas cómo escalar la producción y si hay demanda hospitalaria.
    """),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

regulatorio = Agent(
    role="Experto en Cumplimiento Normativo y Ecotoxicología",
    goal="Garantizar que el producto cumpla normas ambientales y de salud (REACH/EPA).",
    backstory=dedent("""
        Antiguo auditor de la EPA y experto en directrices OCDE.
        Tu misión es identificar barreras legales, riesgos de toxicidad aguda y proponer 
        un plan estricto de mitigación de impacto ambiental.
    """),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

redactor_ejecutivo = Agent(
    role="Redactor Ejecutivo de Negocios",
    goal="Sintetizar hallazgos en un 'Executive Summary' de 1 página sumamente persuasivo.",
    backstory=dedent("""
        Tienes un MBA en Harvard y llevas años armando pitches para inversores.
        Eres maestro en destilar jerga técnica, económica y regulatoria en una 
        propuesta de valor cristalina y convincente.
    """),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

# ==============================================================================
# 3. Definición de Tareas
# ==============================================================================
tema_proyecto = "Comercialización de un recubrimiento antimicrobiano basado en Nanopartículas de Plata (AgNPs) para entornos hospitalarios"

tarea_tecnica = Task(
    description=f"Evalúa la viabilidad técnica de: {tema_proyecto}. Extrae propiedades usando la herramienta y define viabilidad de síntesis.",
    expected_output="Informe de 1-2 párrafos sobre viabilidad técnica y propiedades principales.",
    agent=cientifico
)

tarea_economica = Task(
    description="Con el informe técnico, elabora un análisis de mercado: estima oportunidad económica en hospitales y costos relativos.",
    expected_output="Análisis financiero simplificado y oportunidad de mercado.",
    agent=economista
)

tarea_legal = Task(
    description="Analiza riesgos ecotoxicológicos y barreras legales de las AgNPs. Propón un plan de mitigación sólido para inversores.",
    expected_output="Resumen de barreras legales y estrategia de cumplimiento.",
    agent=regulatorio
)

tarea_resumen = Task(
    description="Integra el trabajo de los otros agentes en un 'Executive Summary' de 1 sola página. Hazlo persuasivo para buscar inversión (VC).",
    expected_output="Un documento 'Executive Summary' persuasivo, estructurado en secciones claras, de 1 página.",
    agent=redactor_ejecutivo
)

# ==============================================================================
# 4. Ensamblar y Ejecutar el Crew
# ==============================================================================
crew_plan_negocio = Crew(
    agents=[cientifico, economista, regulatorio, redactor_ejecutivo],
    tasks=[tarea_tecnica, tarea_economica, tarea_legal, tarea_resumen],
    process=Process.sequential,
    verbose=True
)

print(f"--- INICIANDO GENERACIÓN DE PLAN DE NEGOCIOS: {tema_proyecto} ---\n")
result = crew_plan_negocio.kickoff()

texto_resultado = result.raw if hasattr(result, 'raw') else str(result)
print("\n" + "="*70 + "\nRESULTADO FINAL (Executive Summary)\n" + "="*70)
print(texto_resultado)

# ==============================================================================
# 5. Output Scorer Personalizado
# ==============================================================================
def output_scorer(reporte: str) -> float:
    """Usa un LLM como Juez para calificar la salida de la Crew (0.0 a 1.0)."""
    prompt_score = f"""
    Eres un Inversor de Capital de Riesgo evaluando un 'Executive Summary'.
    Evalúa el reporte bajo los siguientes 5 criterios (suman 1.00):
    1. Base científica y técnica (0.20)
    2. Viabilidad económica (0.20)
    3. Mitigación regulatoria y ambiental (0.20)
    4. Persuasión y claridad (0.20)
    5. Formato de 1 página y coherencia global (0.20)
    
    Reporte a evaluar:
    {reporte}
    
    Responde ÚNICAMENTE con un número flotante entre 0.00 y 1.00. (Ejemplo: 0.92)
    """
    try:
        ans = llm.invoke(prompt_score).content.strip()
        return float(ans)
    except:
        return 0.0

score_final = output_scorer(texto_resultado)
print("\n" + "="*70 + "\nEVALUACIÓN DEL OUTPUT SCORER\n" + "="*70)
print(f"Score obtenido: {score_final:.2f} / 1.00")

if score_final > 0.85:
    print("✅ ¡Éxito! Las backstories de los agentes están óptimamente calibradas y han producido un reporte estelar.")
else:
    print("⚠️ Score bajo. Se sugiere iterar las backstories y roles para mayor precisión.")
